Run the Mirror Quantum Awesomeness experiment as usual.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.transpiler import Target, CouplingMap
from qiskit.quantum_info import Operator
from qiskit.circuit.library import CXGate
from qiskit_device_benchmarking.bench_code.mrb import MirrorQA, QuantumAwesomeness
import os, random, json

SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [2]:
basis_gates = ['id', 'h', 'x', 'y', 'z', 'rz', 'cx']
p2 = 1e-2
p1 = p2 / 10
rz_angle = np.pi / 2

shots = 1000
num_samples = 20
lengths = [2] + [4, 10, 20, 50]

# This can be changed in the future when we want to upgrade the square lattice or use the real IBM machine which has the square lattice.
num_qubits = 16
cmap = CouplingMap.from_grid(4, 4, bidirectional=True)

# Set up the target object (exclusively designed RZ gate)
target = Target.from_configuration(
    num_qubits=num_qubits,
    basis_gates=basis_gates,
    coupling_map=cmap,
    custom_name_mapping={
        "id": Operator(np.array([[1, 0], [0, 1]])),  # Identity gate
        "h": Operator(np.array([[1, 1], [1, -1]]) / np.sqrt(2)),  # Hadamard gate
        "x": Operator(np.array([[0, 1], [1, 0]])),  # Pauli X gate
        "y": Operator(np.array([[0, -1j], [1j, 0]])),  # Pauli Y gate
        "z": Operator(np.array([[1, 0], [0, -1]])),  # Pauli Z gate
        "rz": Operator(
            [
                [np.cos(rz_angle / 2), -1j * np.sin(rz_angle / 2)],
                [-1j * np.sin(rz_angle / 2), np.cos(rz_angle / 2)],
            ]
        ),  # RZ(rz_angle) from above
        "cx": Operator(
            np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])
        ),  # CNOT gate
    },
)

# Set up the noise model
noise_model = NoiseModel()
error_1q = depolarizing_error(p1, 1)
error_2q = depolarizing_error(p2, 2)
for gate in basis_gates:
    if gate in ['id', 'h', 'x', 'y', 'z', 'rz']:
        noise_model.add_all_qubit_quantum_error(error_1q, gate)
    elif gate == 'cx':
        noise_model.add_all_qubit_quantum_error(error_2q, gate)

# Set up the simulation backend (or, in the future, IBM device)       
backend = AerSimulator(
    method="stabilizer",
    noise_model = noise_model,
    target=target,
    max_parallel_threads=0,
    max_parallel_experiments=0,
    seed_simulator=SEED,
)

In [3]:
# The main object to play with
exp = MirrorQA(
    range(num_qubits),
    lengths=lengths,
    backend=backend,
    two_qubit_gate_density=0.25,
    num_samples=num_samples,
    initial_entangling_angle=np.pi / 2,
    sampling_algorithm='new',
    seed=SEED,
)

exp.set_run_options(shots=shots)

In [4]:
rb_data = exp.run()
rb_data.block_for_results()
print("Done.", rb_data.job_ids)

Done. ['94ccc184-1eda-48b2-bb2c-98d589d3f2bc']


Now, we need to extract game data.

In [5]:
qa = QuantumAwesomeness(exp.backend.coupling_map)

# 'oneProb' vs 'sameProb' for the future game
def oneProb(counts, n):
    total = sum(counts.values())
    oneprob = [0.0] * n
    for bs, c in counts.items():
        for q in range(n):
            if bs[-1-q] == '1':
                oneprob[q] += c / total
    return oneprob

def sameProb(counts, edges):
    total = sum(counts.values())
    sameprob = {}
    for q0, q1 in edges:
        if q0 < q1: 
            sameprob[f'{q0}-{q1}'] = sum(
                c / total for bs, c in counts.items()
                if bs[-1-q0] == bs[-1-q1]
            )
    return sameprob

In [6]:
qa = QuantumAwesomeness(exp.backend.coupling_map)
edges = [(q0, q1) for q0, q1 in exp.backend.coupling_map.get_edges() if q0 < q1]
mi = qa.mutual_info(rb_data.data())
n_lengths = len(lengths)

# Read pairs from circuit metadata — NOT from exp._pairs.
# exp._pairs can be stale if any setup cells were re-run after exp.run().
# The metadata is written at circuit-generation time and travels with the results.
pairs_per_sample = [
    rb_data.data()[s * n_lengths]['metadata']['pairs']
    for s in range(num_samples)
]

samples = []
for s in range(num_samples):
    rounds = []
    for i in range(n_lengths):
        idx = s * n_lengths + i
        d = rb_data.data()[idx]
        raw_mi = mi[idx]
        rounds.append({
            'depth': d['metadata']['xval'],
            'oneProb': oneProb(d['counts'], num_qubits),
            'sameProb': sameProb(d['counts'], edges),
            'mi': {f'{q0}-{q1}': round(float(v), 6) for (q0, q1), v in raw_mi.items()},
        })
    samples.append({
        'answers': [list(p) for p in pairs_per_sample[s]],
        'rounds': rounds,
    })

print('Answers of sample 0:', samples[0]['answers'])

Answers of sample 0: [[4, 0], [10, 11], [5, 1], [13, 9], [7, 6], [8, 12], [3, 2], [14, 15]]


## Game Logic: Pair Deduction and Scoring

Given MI values for a round, run **Maximum Weight Perfect Matching** (MWPM) via `rustworkx` to predict which qubit pairs were entangled. Higher MI → more likely a true pair → higher matching weight. This is the same algorithm the original game used (`mwmatching.py`), but now backed by `rustworkx`'s Rust implementation.

In [7]:
import rustworkx as rx

# Fixed 4×4 grid layout positions — used for visualization and future web JSON
GRID_POS = {q: (q % 4, -(q // 4)) for q in range(16)}

def deduce_pairs(mi_dict, coupling_map, n_qubits):
    """Use MWPM on MI values to predict the true qubit pairs.

    Higher MI on an edge = more likely that edge was a true CX pair.
    max_cardinality=True forces as many pairs as possible (same intent as the
    original game's maxcardinality=True in mwmatching.py).
    """
    G = rx.PyGraph()
    G.add_nodes_from(range(n_qubits))
    for q0, q1 in coupling_map.get_edges():
        if q0 < q1:
            w = mi_dict.get(f'{q0}-{q1}', 0.0)
            G.add_edge(q0, q1, w)
    matched = rx.max_weight_matching(G, max_cardinality=True, weight_fn=lambda e: e)
    return [sorted(pair) for pair in matched]

def score_guess(predicted, answers):
    """Returns (correct_count, total_pairs)."""
    pred_set = {frozenset(p) for p in predicted}
    true_set = {frozenset(p) for p in answers}
    return len(pred_set & true_set), len(true_set)

In [8]:
import matplotlib.cm as cm

def draw_game_round(mi_dict, coupling_map, answers, depth, n_qubits=16):
    """Draw the game board: coupling map with MI-colored edges.

    Edge color: green (high MI) → red (low MI).
    Thick edges: MWPM-predicted pairs.
    Numbers on nodes: qubit index.
    Numbers on edges: MI value (the player's clue).
    """
    predicted = deduce_pairs(mi_dict, coupling_map, n_qubits)
    correct, total = score_guess(predicted, answers)
    pred_set = {frozenset(p) for p in predicted}

    fig, ax = plt.subplots(figsize=(7, 7))
    norm = plt.Normalize(0, 1)
    cmap_edges = cm.RdYlGn

    undirected_edges = [(q0, q1) for q0, q1 in coupling_map.get_edges() if q0 < q1]
    for q0, q1 in undirected_edges:
        w = mi_dict.get(f'{q0}-{q1}', 0.0)
        x = [GRID_POS[q0][0], GRID_POS[q1][0]]
        y = [GRID_POS[q0][1], GRID_POS[q1][1]]
        lw = 6 if frozenset([q0, q1]) in pred_set else 1.5
        ax.plot(x, y, color=cmap_edges(norm(w)), linewidth=lw, zorder=1,
                solid_capstyle='round')
        mx, my = (x[0] + x[1]) / 2, (y[0] + y[1]) / 2
        ax.text(mx, my, f'{w:.2f}', ha='center', va='center', fontsize=7,
                bbox=dict(facecolor='white', alpha=0.75, pad=1, edgecolor='none'))

    for q in range(n_qubits):
        x, y = GRID_POS[q]
        ax.scatter(x, y, s=700, color='steelblue', zorder=2,
                   edgecolors='navy', linewidth=1.5)
        ax.text(x, y, str(q), ha='center', va='center', color='white',
                fontsize=9, fontweight='bold', zorder=3)

    plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap_edges), ax=ax,
                 label='Mutual Information', shrink=0.7)
    ax.set_title(
        f'depth={depth}  |  MWPM auto-solver: {correct}/{total} correct\n'
        f'thick edges = predicted pairs  |  numbers = MI clue',
        fontsize=10,
    )
    ax.set_aspect('equal')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Draw all difficulty rounds for sample 0
for r in samples[0]['rounds']:
    draw_game_round(r['mi'], exp.backend.coupling_map, samples[0]['answers'], r['depth'])

TypeError: 'float' object cannot be interpreted as an integer

In [ ]:
# Export all game data to JSON for the future web frontend.
# The backend sends this payload; the frontend renders nodes/edges and accepts player input.
# 'answers' (true pairs) stay server-side for scoring — strip them before sending to the client.

game_export = {
    'config': {
        'num_qubits': num_qubits,
        'grid_rows': 4,
        'grid_cols': 4,
        'lengths': lengths,
        'num_samples': num_samples,
        # str keys required for JSON; values are [x, y] grid coordinates
        'grid_positions': {str(q): list(pos) for q, pos in GRID_POS.items()},
        'coupling_edges': [[q0, q1] for q0, q1 in exp.backend.coupling_map.get_edges() if q0 < q1],
    },
    'samples': samples,  # each: {'answers': [...], 'rounds': [{depth, mi, oneProb, sameProb}, ...]}
}

with open('game_data.json', 'w') as f:
    json.dump(game_export, f, indent=2)

print(f"Saved {num_samples} samples to game_data.json")
print(f"Keys per sample: {list(samples[0].keys())}")
print(f"Keys per round:  {list(samples[0]['rounds'][0].keys())}")

In [ ]:
# Auto-solver accuracy across all depths for every sample.
# This shows how hard each depth level is — the game's difficulty curve.
print(f"{'sample':>7}  {'depth':>5}  {'result':<20}  score")
print("-" * 50)
for s, sample in enumerate(samples):
    for r in sample['rounds']:
        predicted = deduce_pairs(r['mi'], exp.backend.coupling_map, num_qubits)
        correct, total = score_guess(predicted, sample['answers'])
        bar = '█' * correct + '░' * (total - correct)
        print(f"  s={s:>2}    d={r['depth']:>3}    [{bar}]   {correct}/{total}")